# 实验七：大语言模型实验

本 Notebook 对应 `exp7_llm_rag_agent.py`，包含三部分：提示工程、RAG 企业政策问答、简易 Agent 模拟。

默认使用 `mock` 后端，无 API Key 也能先跑通；正式实验时把 `backend` 改成 `zhipuai`、`deepseek` 或 `openai`，并在 `.env` 中配置 API Key。

本实验不训练本地大模型，只做 API 调用、文本向量化和 FAISS CPU 检索，因此普通 PC 即可运行。


In [ ]:
# 依赖安装示例（需要时取消注释执行）
# %pip install -U openai zhipuai python-dotenv pypdf faiss-cpu numpy pandas

from pathlib import Path
from IPython.display import Markdown, display


In [ ]:
from exp7_llm_rag_agent import (
    ExperimentConfig,
    ChatClient,
    EmbeddingClient,
    run_prompt_engineering,
    run_rag_experiment,
    run_agent_experiment,
    set_seed,
)

set_seed(42)

# mock：离线演示；zhipuai：智谱AI；deepseek：DeepSeek 聊天生成；openai：OpenAI 或兼容接口；auto：自动检测 API Key。
backend = 'mock'

cfg = ExperimentConfig(
    backend=backend,
    output_dir=Path('outputs_exp7'),
    kb_dir=Path('knowledge_base_exp7'),
    rebuild_kb=True,
    top_k=4,
)

chat_client = ChatClient(backend=cfg.backend, temperature=cfg.temperature, max_tokens=cfg.max_tokens)
embedding_client = EmbeddingClient(backend=cfg.backend, dimensions=cfg.embedding_dim)

print('LLM 后端：', chat_client.backend, chat_client.model)
print('Embedding 后端：', embedding_client.backend, embedding_client.model)


## 第一部分：提示工程实践


In [ ]:
prompt_result = run_prompt_engineering(chat_client, cfg.output_dir)
display(Markdown(Path(prompt_result['markdown_path']).read_text(encoding='utf-8')))


## 第二部分：RAG 系统构建


In [ ]:
rag_result = run_rag_experiment(chat_client, embedding_client, cfg)
display(Markdown(Path(rag_result['markdown_path']).read_text(encoding='utf-8')))


## 第三部分：简易 Agent 设计


In [ ]:
agent_result = run_agent_experiment('人工智能在医疗领域的应用', cfg.output_dir)
display(Markdown((cfg.output_dir / 'exp7_agent_trace.md').read_text(encoding='utf-8')))


## 输出文件检查


In [ ]:
for path in sorted(cfg.output_dir.glob('*')):
    if path.is_file():
        print(path)
